In [1]:
import pandas as pd
import numpy as np

In [7]:
df_raw = pd.read_csv("data/raw/interval_data.csv")
df = df_raw.copy()

In [8]:
df.head()

,season,team,interval,shots,goals,xG,shots_against,goals_against,xGA
0,2021,Arsenal,1-15,81,12,9.510792,53,6,5.800125
1,2021,Arsenal,16-30,97,14,11.954485,61,5,4.698751
2,2021,Arsenal,31-45,97,5,10.802890,62,9,6.827177
3,2021,Arsenal,46-60,104,14,11.128576,77,9,8.016165
4,2021,Arsenal,61-75,101,5,10.094803,76,9,11.756935


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   season         180 non-null    int64  
 1   team           180 non-null    str    
 2   interval       180 non-null    str    
 3   shots          180 non-null    int64  
 4   goals          180 non-null    int64  
 5   xG             180 non-null    float64
 6   shots_against  180 non-null    int64  
 7   goals_against  180 non-null    int64  
 8   xGA            180 non-null    float64
dtypes: float64(2), int64(5), str(2)
memory usage: 12.8 KB


In [10]:
# 得点数からxG（期待値）を引いた値　「決定力の指標」
df["goals_minus_xG"] = df["goals"] - df["xG"]
# 実際の失点からxGA（期待値）を引き算した「ミス・集中力キレの指標」
df["goals_against_minus_xGA"] = df["goals_against"] - df["xGA"]


In [11]:
# 各シーズン・チームごとの総得点・失点数を計算して追加
df["total_goals"] = df.groupby(["season","team"])["goals"].transform("sum")
df["total_goals_against"] = df.groupby(["season", "team"])["goals_against"].transform("sum")

# 時間帯別の得点・失点割合（%）を計算
df["goals_percentage"] = (df["goals"] / df["total_goals"]) * 100
df["goals_against_percentage"] = (df["goals_against"] / df["total_goals_against"]) * 100

In [19]:
cols_order = ["season", "team", "interval",
              "shots", "goals", "xG", "total_goals", "goals_minus_xG", "goals_percentage",
             "shots_against", "goals_against", "xGA", "total_goals_against", "goals_against_minus_xGA", "goals_against_percentage"]

In [20]:
df[cols_order].to_csv("data/processed/cleaned_interval_data.csv", index=False)

In [21]:
df[cols_order].head()

,season,team,interval,shots,goals,xG,total_goals,goals_minus_xG,goals_percentage,shots_against,goals_against,xGA,total_goals_against,goals_against_minus_xGA,goals_against_percentage
0,2021,Arsenal,1-15,81,12,9.510792,61,2.489208,19.672131,53,6,5.800125,48,0.199875,12.500000
1,2021,Arsenal,16-30,97,14,11.954485,61,2.045515,22.950820,61,5,4.698751,48,0.301249,10.416667
2,2021,Arsenal,31-45,97,5,10.802890,61,-5.802890,8.196721,62,9,6.827177,48,2.172823,18.750000
3,2021,Arsenal,46-60,104,14,11.128576,61,2.871424,22.950820,77,9,8.016165,48,0.983835,18.750000
4,2021,Arsenal,61-75,101,5,10.094803,61,-5.094803,8.196721,76,9,11.756935,48,-2.756935,18.750000


In [ ]:
# 均等に失点した場合の基準値（100％ ÷ 6グループ）
base_percentage = 100 / 6
# 基準値からのズレを計算
df["conceded_deviation"] = df["conceded_percentage"] - base_percentage